In [1]:
# Install necessary libraries
!pip install transformers datasets pandas scikit-learn torch

# Import required libraries
from google.colab import files
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils import resample
from transformers import EarlyStoppingCallback# Step 1: Upload and Load Dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:


# Step 1: Upload and Load Dataset
uploaded = files.upload()  # Upload the dataset file
DATA_FILE = list(uploaded.keys())[0]  # Get the uploaded file name
DATA = pd.read_csv(DATA_FILE)
DATA['label'] = DATA['binary_label'].astype(int)
DATA['text'] = DATA['text'].str.replace(r"https:\/\/t.co\/\S+", "[URL]", regex=True)

# Step 2: Balance the dataset
# Oversample the minority class
minority_class = DATA[DATA['label'] == 1]
majority_class = DATA[DATA['label'] == 0]

oversampled_minority = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=42
)

balanced_data = pd.concat([majority_class, oversampled_minority])

# Step 3: Train-Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced_data['text'], balanced_data['label'], test_size=0.2, stratify=balanced_data['label'], random_state=42
)

# Step 4: Compute Class Weights
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_labels), y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# Step 5: Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Step 6: Tokenizer and Base Model
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Custom model class to include class weights and dropout
class WeightedBERT(torch.nn.Module):
    def __init__(self, model, class_weights):
        super(WeightedBERT, self).__init__()
        self.model = model
        self.dropout = torch.nn.Dropout(0.4)  # Increased dropout for regularization
        self.register_buffer("class_weights", class_weights)

    def forward(self, input_ids, attention_mask, labels=None):
        device = input_ids.device
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)
        return {"loss": loss, "logits": logits}

# Load Pretrained BERT Model and Wrap It
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = WeightedBERT(base_model, class_weights)

# Step 7: Tokenize the Dataset
def tokenize_function(sample):
    return tokenizer(sample['text'], truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Step 8: Data Collator for Dynamic Padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 9: Define Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Step 10: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    num_train_epochs=5,  # Early stopping will terminate training earlier
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.2,  # Increased weight decay for stronger regularization
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    lr_scheduler_type="cosine_with_min_lr",  # Use cosine scheduler with min LR
    warmup_steps=500,
    lr_scheduler_kwargs={"min_lr": 1e-6},  # Specify the minimum learning rate
    report_to=[],
)


# Step 11: Initialize Trainer with Early Stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # Early stopping
)

# Step 12: Train the Model
trainer.train()

# Step 13: Evaluate the Model
results = trainer.evaluate(test_tokenized)
print("Test Results:", results)


Saving DefaktS_Twitter.binary.csv to DefaktS_Twitter.binary.csv


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Map:   0%|          | 0/18833 [00:00<?, ? examples/s]

Map:   0%|          | 0/4709 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.475000,0.379642,0.829900,0.827035,0.841019,0.813509
2,0.407600,0.350387,0.840306,0.821801,0.929260,0.736619
3,0.331800,0.338115,0.873646,0.871463,0.886593,0.856839
4,0.276400,0.359877,0.890210,0.891318,0.882231,0.900595
5,0.270500,0.413191,0.891484,0.892489,0.884118,0.901020


Test Results: {'eval_loss': 0.41319096088409424, 'eval_accuracy': 0.8914843915905712, 'eval_f1': 0.8924889543446245, 'eval_precision': 0.8841183826594414, 'eval_recall': 0.9010195412064571, 'eval_runtime': 14.2527, 'eval_samples_per_second': 330.395, 'eval_steps_per_second': 20.698, 'epoch': 5.0}
